## CASH Notebook

## Celestial Chase - LVL 1: The Teal Beacon

You've just woken up from cryo-sleep. No memory. No crew. Just you, a dying sun, and a faint signal pulsing from the sky.

One star glows different from the rest. Not white. Not grey. **Teal.**

Find it. Its position holds part of the key. But position alone is not enough - you must also know how much of its face is lit by the sun.

---

**The encoded signal:** `emulod`

**Your task:**
1. Display the image and find the **cyan pixel** - it is the only pixel where `B == 255` and `G == 255` and `R == 0`
2. Read its `(x, y)` coordinates
3. Use `ephem` to compute **Venus's phase** (`int(venus.phase)`) on `2025/6/21 12:00:00` UTC from Zurich (lat=`47.3769`, lon=`8.5417`)
4. Build the key: `key = x * y + int(venus.phase)`
5. Build the permutation: `random.Random(key).shuffle(list(range(len(encoded))))`
6. Reverse the transposition to decode: `decoded[i] = encoded[perm[i]]`


In [ ]:
import base64, cv2, numpy as np
from IPython.display import display, Image as IPImage

# Starfield image embedded directly in this notebook
img_b64 = "iVBORw0KGgoAAAANSUhEUgAAAZAAAAGQCAIAAAAP3aGbAAAUhklEQVR4Ae3BZ1TWd5738c/32T6azWZzT4ppY0nZTIwmubOT2EZHgyKKSLGhAhpRxMagiIJGQRFlsCGKUUDFRhFRRFFHR8VkNptEYyabYpk0U+bOZnPPo33GnpM9nmNOQkK5yv93Xe/Xy4SQULavKnViojxsw85X5k97SUAXmAB0WUHJpuz0uYKfmX7OuOkpB3eUK4TMXPj7bev+ILjmxdiYk3X1QhgzBVbu2jV5ixYLbUjNzCgrKpY/pWdnlRQUCp0yZfas3Vu2CkFiws+JT0mqKa8UQtHVLz/vdc99Qmf9x6cf/8sDDylQTADgCBMAOMIEAI4wAYAjTMCPGTN50uE9e/V946anHNxRLiDgBo+OOnuk0QR03F+//tuv7vqlENIaz52NGjRYgbK7vm5KTKx+kgkAHGEKrP4jIlqONwsAOs4UJOfeemPQ088KANrNBECKHBffdLBG8DYTADjCFDxpWQtLC9cJCDktb1/q/1RfwddMAELa+h3bF0yfoZBgQkAMioo819gkAF1gAryk4czp6CFDBfwYE9A1WfkrC3OWCfA/EwA4wgQAjjABXlVYWpKVli7gFhMAOMIE3Gb7/r0zJkwSwk/+xvU58xbI20yA/1UfP5YwYqRCy0sZ818p3iAEkAkIJx9/8/VDd94luMkEAI4wAYAjTADgCBPCTGzSlLrK3QJ85PL1q3169FJAmICAW7etdOHMNAEdZAIAR5hcMDQm+nR9gwB02cfffP3QnXfJTSYAQTIiIe54da3QbiZ41cjxCccOVAvALSYgPGzZXTl7SpLgMhMA1xSVbc1MnaW25W0ozp2foZBjQjhpPHc2atBgAW4yAYAfnH3j9cHPPiefMoW3G//vq+7/524BcIEJABxhAgBHmADAEaZQ0fL2pf5P9RWA0GUCPKCqoT4xOkbATzIBgPc0v9oS8UJ/fZ8JABxhus20+XN3btiksHflr9d7/6qHAPykj7/5+qE771IAmQDAESYAjohPSaopr1QYMwHwjOHxsSdq6oQ2mAB0zYFjR8ePHCX4n8kFg0dHnT3SKOAHMpbnFq/IE3zt0rUP+/Z8RB5jAoKh/vTJmKEvKvyMm55ycEe50CkmAHCECQAcYQIAR5gAwPPOX3pzYN9nTJ5Xd/JE7IvDBSDsmQDgll+2tv7NTF5lgmve/PD9Zx55TED4MQHwtcvXr/bp0UvwNVOgzM9duiFvlQCgs0wA4AgTADjCBACOMAGAI0yuOXSqeeywCAHB88+trf9pppBw4uKF4f0GyBEmAHCECQAcYQIAR5gAz/vj63/+3XO/EcKeyW+O/unMqN8OUZdFjotvOlgjAGHPBACOMAGAI0yAL8xZsnjz6jUC/Ml0m7VbtyyaNVsA3BebNKWucrdCiwkAHGHqsr1HDk8aPUZtGBoTfbq+QQDQZSYAcIQJ8LVL1z7s2/MRoYOKyrZmps4SbjN60oQje/frFhO+b++Rw5NGjxEA7zEB8IWyfVWpExMFfzIhpGUszy1ekScEW82JpvjhkfpJNSea4odHCm0zfWd50doVmYsEAB5mAoDOWrwqb83SXAWKCQAcYQK6rGxfVerERAF+Zgo/EXFjm2sPyT+qGuoTo2METxo7dfKhXXsEZ5kC7t1PPnriwYcFAB1kAgBHmAAgUBaufHndspfVWabgKSjZlJ0+VwDQPiZpQOTwC00nBMBxk2bO2Lttu0KXCSGk/4iIluPNAkKUKbQ0v9oS8UJ/AQhFJgBoW8vbl/o/1VfeYAIAR5gQJOnZWSUFhQLQbiYAcIQJABxhAgBHmNC2LbsrZ09JUtg4/MdTY343TIBXmb5vU8XOucnTBADeYwq21MyMsqJiecm+ow0TR0UL+IEFy3LWr8wXOqXmRFP88Eh1gQkAHGGCVNt8PC5ihAB4mwnwp2tffdHz7nsVEsa/NO3AKzuF4DG5KSJubHPtIcF3cgoL8rOyBXiYCW078+//NuT//qsAeIMpPHz8zdc3btwY/OxzAuAsEwA4whQMKfPmlG/cLLQhPiWpprxSQNtejI05WVevMGMCgiqnsCA/K1tAO5gAwBEmAGjb2zeuPdW9p7zBBCBUzF68aMuatQpdJo85dKp57LAIAfCR7fv3zpgwSSHBBACOMAEe0HDmdPSQoQJ+kgkAHGFCYEXEjW2uPaTgqWqoT4yOEXyrtVVmgp+ZAHRRa6v+l5ngTyY4KCt/ZWHOMsE7WltlJvjHuOkpB3eUSzIB8KfxL0078MpOwRdMIWTxqrw1S3MF4Jajfzoz6rdD1Ib07KySgkK5wwTABSW7KtKnJiu8mQCEhP4jIlqONyukmQDglq1Vu2clTpFXmQD4WumeXWmTp8ozdhzcP33cBLnPBB/Zsrty9pQkAfAbEwA4wgR4WFVDfWJ0jNAFsxZlbl1bpJBgAgBHmH5ObNKUusrdCpJrX33R8+57BQCSCQAcYQIAR5g65eKVy/169xEAX1i9eeOSOfN0y/lLbw7s+4zwAyYAcIQJABxh8rYPPv/s0fvuF9ohb0Nx7vwMdcTRP50Z9dshAhxh+kk5hQX5WdlCWIpOnNhQtU/4SXUnT8S+OFwBV1FbnRyXoADaWX1gWsJ4+cfG8h3zUqbr55gA4DZDY6JP1zfIk0wA4AgTQt3AkSPOHzsuBMq1r77oefe9gh+YACCwml9tiXihvzrOBACOMAGAI0wA4AgTADjCBACOMHXNkoJVq7OXCgD8zwR0RM2JpvjhkeqCdz668eTD3RVOUubNKd+4WegyEwA4wgSgHQ42NY6LjBKCygQHFb9SlvFSqoAwYwL86f2bnz7W7QEBvmACAEeY4AdLClatzl4qAD5lAgAPi4gb21x7SN8xAUCgnH3j9cHPPqfOMgFS/xERLcebBXibycNe+8uV53/dWwA848Lltwb0eVpBYgIAR5gQPNPmz925YZPgH9v3750xYZLQcSnz5pRv3CzvMQGAI0xeUllXkxQbL7RDRW11clyCgHBiglReczAlfpwAeJsJADpiY/mOeSnTFQwmAM4aOT7h2IFqhQ0TADjC1AUHmxrHRUYJgAtipiTW766Sj0xNT9tVUqrAMkknX7v44vP9BNxmRXHR8oxMocvikqfWVuwSfMEUQGX7qlInJgoICWu2bF48e44QQCYg/MzLWbIxf7XgGhMAOMKEn1TVUJ8YHSMAHmBySmpmRllRsTprzORJh/fsFeBrFbXVyXEJgp+Z/GNoTPTp+gYBISQueWptxS4heEzS5etX+/ToJV/4t//4y7/+y68FAH5gAgBHmAB0yrCxY04dOiwEkMlNFbXVyXEJ6qDpC+btWL9RnbW7vm5KTKzCxqSZM/Zu266fs3Dly+uWvSygHZYUrFqdvVSdZQqghGnJ1TsrBACdYgIAR5gQlhYsy1m/Ml9wx87qA9MSxstBJbsq0qcmyxdM+IE1WzYvnj1HbhoaE326vkFAKDIB6Lj4lKSa8kohsEwIJyvX/2HZgt8LcJMJABxhQjB8+u03D9xxp7osp7AgPytboeudj248+XB33TI0Jvp0fYMQrkwA4AgT4Asv/2Hdy79fKMCfTAgDsxZlbl1bJMBxJqDjCko2ZafPFTzmntbWL80UukwA4AgTEKKmpqftKikV/O+Dzz979L77dUta1sLSwnXyAxOAcPXqO2+/8ORTarfjLedH9B+o71y4/NaAPk/LnxaufHndspd1GxMAOMIEAI4wAYAjTED7XP3y81733CfARy5eudyvdx91hAnwnreufvB0r0cFfJ8JADzvzQ/ff+aRx0wA0BEvxsacrKtXMJgQJPNzl27IWyUPe/29d597/AkBnmECAEeYfGp+7tINeasEAL42eHSUCaFlyuxZu7dsFcLPW1c/eLrXowppJqCzbv79226/uENAoJgAwBEmtNvu+ropMbH6TkVtdXJcgrztrtbWr80EX5ifu3RD3iqhCxYsy1m/Ml/Slb9e7/2rHuo4k1c1XTgXOWCQEPaOt5wf0X+gAMkEAI4wAYAjTIDLhkSPOtNwVGGs8dzZqEGDFR5MANAphaUlWWnpCiATADjCBKAjcgoL8rOyhWAw+VT18WMJI0YKAPzABACOMDkiZd6c8o2bBSCMmeAZFbXVyXEJQljKXp1fsCRHXfPaX648/+veCl0mAHCECUHSdOFc5IBBAm6TlrWwtHCd0AYT8J2LVy73691HQAet2rRh6dz5CgjTLbMWZW5dWyQ/ePWdt1948ikB8KnGc2ejBg1WODEBCKxP/us/H/ynfxY6ziQNHDni/LHjAjrr+t++7PHLewT4mckpvxsz+o+HjwjhZ+X6Pyxb8HvBG0aOTzh2oFoBZ4Ljtu3dM3PSZAFhwAQAjjABCKzKupqk2Hih40wd9+d33/nNE08KAALLBACOMAHwm0FRkecamwQfMQGAI0z4ztUvP+91z30C4GEm/JjIcfFNB2sE73nnoxtPPtxdCJRrX33R8+575Q0mAB1UWFqSlZYuBJwJ4S1pzuzKzVsEuMAEuOPC5bcG9HlaCFcmAHCECQAcYQIAR5gAwBEmAHCECQB8bWhM9On6BvmaCQAcYQpXk9Nm7indJgDuMAGAI0w/sL/xyISo0QpRk2bO2LttuwIuclx808EahZyPv/n6oTvv0i03//5tt1/cIcA/TADgCBMAOMIEAI4wAYAjTEBgHWxqHBcZJaDjTCGtsLQkKy1dAEKCCQAcYQqgyHHxTQdrBMCnUjMzyoqKFQZMYS8ueWptxS4B8DwTADjCBACOMAGAI0wIby1vX+r/VF8hzMxdmr1pVYFcYwIQbO98dOPJh7sLP8cEtMPNv3/b7Rd3yEG76+umxMQKIcEE9+2sPjAtYbyAUGcCvGfu0uxNqwrUPsWvlGW8lKp2y1yxvGj5CnnMiYsXhvcboPapO3ki9sXhCj8mAHCECQiqa1990fPuewW0gwkAHGHymztaW78104+pOdEUPzxScMesRZlb1xYJHvbaX648/+veCmkmIPxcuvZh356PCK4xdcTVLz/vdc99AhyRPDe9YlOJECpMAKTs1fktLS0Xmk4IHmaCl9z8+7fdfnGHEKIylucWr8iTm1IzM8qKihVUprCXsTy3eEWeAATcOx/dePLh7mo3E0LOiuKi5RmZAkKOCQAcYQIAR5gAdMrSNatXLV4iBJAJQJfd1dr6tZnc9O4nHz3x4MNygQkAfCpmSmL97ir5gen7Tv351WG/eUEIlPc+++Tx+x8UgHYwAYAjTIAvDI+PPVFTJ8CfTIFVtq8qdWKiws+hU81jh0UIQBeYAG9495OPnnjwYQFtMwHwtiNn/zh68O8EyQTAn96/+elj3R4QfMEE3/mH1tb/NlNIGzhyxPljxwUEg+nnxCVPra3YJcBxn/3//7r/H/9JcJkJ8LbMFcuLlq9QMKzevHHJnHmCZ5gAwBEmAHCECQAcYULYS8/OKikoFOB5JrStorY6OS5B6IITFy8M7zdAgC+YAMARJgBBsn7H9gXTZwjtZoKb9jcemRA1WvCYSTNn7N22XeEtf+P6nHkL5AcmAHCECQERn5JUU14pAF1ggoNqTjTFD48UEGZMAOAIE4Aw8OEXNx+5t5vaJ3ftmrxFi+U9JgBwhAkuSJyVWrW1TAgbpXt2pU2equAp21eVOjFRHmPCLeu2lS6cmSYAXmWCl3zw+WeP3ne/APwYEwA4wgS47/L1q3169FJoGRQVea6xSb4zOW3mntJtcpkJABxhAhB+rn31Rc+775VrTHDc+h3bF0yfISAMmADAESbc5o0P3nv20ccFwJNMAOAIEwA4wgS0YcfB/dPHTRDgGSaEq/U7ti+YPkPhJ3t1fsGSHMFBJgDolNikKXWVuxVAJgBwhMmnBo+OOnukUQDgByZ4xuXrV/v06CUAbTABoWJoTPTp+gYhdJkAN1U11CdGx8g1FbXVyXEJChX7jjZMHBWtQDF5w+hJE7p3774hb5U64v2bnz7W7QEBvlBUtjUzdZb849Cp5rHDIoSuMQGAI0wA4AgTADjCBIS9gpJN2elzBc8zoW0VtdXJcQkC4GcriouWZ2Tq55gAwBEmwBcOHDs6fuQoAf5kAgBHmADAESYAXTByfMKxA9VCQJgAwBEmAOFhUFTkucYmucyEUDFs7JhThw4LCF2mUJGxPLd4RZ4AhC4TgBD1/s1PH+v2gEKICQAcYQIAR5g6bu+Rw5NGjxHQQZevX+3To5eAzjIBgCNMAOAIE4D22d94ZELUaHlbxvLc4hV5ClEmAHCEqR2O/unMqN8OkbTrUO3UsXECHLFuW+nCmWlCqDABgCNMuCV7dX7BkhzBcdmr8wuW5AihyAQggC5eudyvdx+hU0wA4AgTADjCBG+LT0mqKa8UAMnkjuMt50f0HygA4coEj8kpLMjPyhbgH/EpSTXllXKTCf5U23w8LmKEAL8pLC3JSktXeDD5065DtVPHxgmhKHftmrxFiwUEkAkAHGEC4KbSPbvSJk9VODEBgCNMQJjJWJ5bvCJPcJDJSxrOnI4eMlTwhn1HGyaOihbgGSYAzopNmlJXuVthw/Sd9z775PH7HxQAeJgJCGnp2VklBYVCSDABgCNMbpq7NHvTqgJ1ytk3Xh/87HMC4BoT0A79R0S0HG8WEFQmuK90z660yVMFhDoTADjChND1Usb8V4o3CAgVJgC4TXTixIaqffIkEwA4wnSbA8eOjh85SoCvRU0Y17j/oOCCc2+9MejpZ+VJJgCuOXHxwvB+AxR+TADgCBNcMy9nycb81YIHjJk86fCevUKgmADAESYA7TAx9aV9Za8IQWUCAEeYAMARpk5JnJVatbVMABBAJgBwhCl4zl96c2DfZwQA7WMCAEeYPObTb7954I47BQA/YAoJBSWbstPnCl0wL2fJxvzVAjzMBADt88YH7z376OMKHlNg7aw+MC1hvLxq9eaNS+bMEwBPMgGAI0xh4ORrF198vp+A4Dn151eH/eYFoWtMfnboVPPYYREKP5V1NUmx8fKp+blLN+StEhCuTAi4jOW5xSvyBKCD/gcuYxu3IoKRXQAAAABJRU5ErkJggg=="

img_bytes = base64.b64decode(img_b64)
img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
img       = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

cv2.imwrite('starfield.png', img)
display(IPImage('starfield.png'))
print("Image shape:", img.shape)


In [ ]:
import ephem, random

encoded  = "emulod"
obs_date = "2025/6/21 12:00:00"
obs_lat  = "47.3769"
obs_lon  = "8.5417"

# TODO Step 1: Find the cyan pixel (B=255, G=255, R=0) in img
# Hint: use np.where or a loop
pixel_x, pixel_y = 0, 0  # replace with actual coordinates

# TODO Step 2: Compute Venus phase with ephem
obs = ephem.Observer()
obs.lat  = obs_lat
obs.lon  = obs_lon
obs.date = obs_date
venus = ephem.Venus()
venus.compute(obs)
phase = 0  # replace: int(venus.phase)

# TODO Step 3: Build key and permutation
key  = pixel_x * pixel_y + phase
perm = list(range(len(encoded)))
random.Random(key).shuffle(perm)

# TODO Step 4: Reverse the transposition
# Hint: if encoded[perm[i]] = original[i], then decoded[i] = encoded[perm[i]]? Think carefully.
def transpose_decode(encoded, perm):
    pass  # implement this

answer = transpose_decode(encoded, perm)
print(answer)
